# Notebook 43 · Asymmetric tail risk and expected shortfall research

**Research-only; zero transaction costs; development data only.** This notebook tests whether a joint model of upside and downside risk can improve upon Ridge α100 without merely selecting volatile stocks. The final 20% historical holdout and prospective ledger remain closed. All portfolio results are out-of-sample chronological folds, but **model selection across these folds is exploratory**.

**Methodological limits:** Daily close-to-close labels do not identify intraday stops or true trade-level expected shortfall; conditional downside estimates below are proxies. No threshold is optimized using validation labels.

## Experiments 1–5 · Load the frozen development universe and build features

In [ ]:
from pathlib import Path
import sys,time
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
assert (ROOT/'src').is_dir(),f'Run from repository root or notebooks/, not {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.classifier_benchmark import eligible_universe
from stock_ml_dynamic.feature_ablation import development_holdout_split
from stock_ml_dynamic.historical_market import add_historical_market_features
from stock_ml_dynamic.accuracy_features import add_accuracy_features
from stock_ml_dynamic.volume_regime_research import add_volume_regime_features
from stock_ml_dynamic.tail_edge_42 import *
from stock_ml_dynamic.edge_robustness_41 import fit_and_score,run_portfolio_stress
OUT=ROOT/'artifacts/tail_edge/1d';OUT.mkdir(parents=True,exist_ok=True)
master,latest,bundle,manifest=load_master(ROOT/'artifacts/master_tables/1d')
print('Master:',master.shape,'Inference rows:',latest.shape,'Output:',OUT)
from stock_ml_dynamic.asymmetric_edge_43 import run_research, paired_comparison, event_reliability, concentration, save_tables
OUT=ROOT/"artifacts"/"asymmetric_edge"/"1d"

In [ ]:
eligible,eligibility=eligible_universe(master)
development,cutoff=development_holdout_split(eligible,.20)
expanded,historical=add_historical_market_features(development)
expanded,technical=add_accuracy_features(expanded)
expanded,features=add_volume_regime_features(expanded)
print('Development:',expanded.shape,'Feature extras:',len(features),'Holdout cutoff:',cutoff)
assert len(expanded)>0 and len(features)>0

## Experiments 6–10 · Reconcile Ridge α100 with Notebook 41 before any new comparisons

In [ ]:
from stock_ml_dynamic.edge_robustness_41 import fit_and_score,run_portfolio_stress
old_scored,_,old_audit=fit_and_score(expanded,features,cap=60000,seed=42)
old_results=run_portfolio_stress(old_scored)
old_base=old_results['metrics'].loc[(old_results['metrics']['model']=='ridge_all') & (old_results['metrics']['k']==5),['fold','mean_daily']]
display(old_base.sort_values('fold'))

## Experiments 11–25 · Joint positive/negative tail classifiers: ±3%, ±5%, ±10%, linear and nonlinear

In [ ]:
import time
start=time.perf_counter()
results=run_research(expanded,features,cap=60000,seed=42)
print('Elapsed seconds:',round(time.perf_counter()-start,1))
scored=results['scored'];metrics=results['metrics'];daily=results['daily'];trades=results['trades']
print('Model variants:',scored.shape[1]-4,'Portfolio evaluations:',len(metrics))
print('Training audit:');display(results['audit'])
assert results['audit'].training_label_end.lt(results['audit'].validation_start).all()
new_base=metrics.loc[(metrics['model']=='ridge_all')&(metrics['k']==5),['fold','mean_daily']]
reconcile=old_base.merge(new_base,on='fold',suffixes=('_41','_43'))
reconcile['absolute_difference']=(reconcile.mean_daily_41-reconcile.mean_daily_43).abs()
display(reconcile)
assert reconcile.absolute_difference.max()<1e-8,'Baseline mismatch: stop model comparisons' 

## Experiments 26–31 · Directional information vs generic volatility: ROC-AUC, AP, Brier, prevalence

In [ ]:
ev=results['events']
display(ev.groupby(['side','threshold','model'],as_index=False).agg(auc=('auc','mean'),average_precision=('average_precision','mean'),prevalence=('validation_prevalence','mean'),brier=('brier','mean')).sort_values(['threshold','side','auc'],ascending=[True,True,False]).round(5))
for t in (.03,.05,.10):
    subset=ev.loc[ev.threshold.eq(t)]
    print(f'Threshold ±{int(t*100)}%:')
    display(subset.pivot(index='fold',columns='model',values='auc').round(4))

## Experiments 32–36 · Fixed-bin probability reliability; inspect upside/downside overconfidence

In [ ]:
calibration=event_reliability(scored)
summary_cal=calibration.groupby(['model','bin'],as_index=False).agg(n=('n','sum'),predicted=('predicted','mean'),observed=('observed','mean'))
display(summary_cal.loc[summary_cal.model.isin(['up_5_hgb','down_5_hgb','up_10_hgb','down_10_hgb'])].round(5))

## Experiments 37–49 · Seven conditional quantiles, joint tail utility, asymmetry ratios and downside penalties

In [ ]:
leader=metrics.groupby(['model','k'],as_index=False).agg(mean_daily=('mean_daily','mean'),market_excess=('market_excess','mean'),payoff_ratio=('payoff_ratio','mean'),profit_factor=('profit_factor','mean'),worst_drawdown=('max_drawdown','min'),positive_folds=('mean_daily',lambda x:int((x>0).sum())))
display(leader.loc[leader['k'].eq(5)].sort_values('mean_daily',ascending=False).head(30).round(5))
print('Ridge baseline:');display(leader.loc[(leader.model=='ridge_all')&(leader['k'].eq(5))])

## Experiments 50–56 · Payoff 2:1, profit factor, fold stability and downside control

In [ ]:
selected_names=['ridge_all','tail_direction_5','weighted_tail_utility','tail_utility_5_penalty2','q90_minus_q10_risk','q90_minus_2down','q95_downside_ratio','extreme_volatility_5','extreme_direction_fraction_5']
display(metrics.loc[metrics['k'].eq(5)&metrics.model.isin(selected_names),['fold','model','mean_daily','market_excess','win_rate','avg_win','avg_loss','payoff_ratio','profit_factor','max_drawdown']].sort_values(['model','fold']).round(5))
print('Number of 2:1 payoff successes across all evaluated fold/strategy/position combinations:',int(metrics['payoff_ratio'].ge(2).sum()))

## Experiments 57–62 · Paired moving-block bootstrap against the *reconciled* Ridge baseline

In [ ]:
paired=paired_comparison(daily,k=5,reps=1200,seed=42)
paired_summary=paired.groupby('model',as_index=False).agg(mean_lift=('mean_lift','mean'),positive_folds=('mean_lift',lambda x:int((x>0).sum())),folds_ci_above_zero=('ci_low',lambda x:int((x>0).sum())))
display(paired_summary.sort_values('mean_lift',ascending=False).head(25).round(6))
print('Individual fold confidence intervals for leading contenders:')
display(paired.loc[paired.model.isin(paired_summary.nlargest(5,'mean_lift').model)].sort_values(['model','fold']).round(6))
print('WARNING: These intervals are unadjusted for searching many correlated strategies.')

## Experiments 63–67 · Large-winner concentration, extreme downside, and symbol concentration

In [ ]:
tails=concentration(trades,k=5,fraction=.01)
display(tails.loc[tails.model.isin(selected_names)].sort_values(['model','fold']).round(4))
for name in ['ridge_all','weighted_tail_utility','q90_minus_q10_risk']:
    g=trades.loc[(trades.model==name)&trades['k'].eq(5)]
    print(name,'top 10 symbols by total trade return')
    display(g.groupby('symbol').next_return.agg(['size','sum','mean']).sort_values('sum',ascending=False).head(10).round(4))

## Experiments 68–73 · Large-winner versus large-loser hit rates at top five; directionality versus volatility

In [ ]:
rows=[]
for (fold,model,k),g in trades.loc[trades['k'].eq(5)].groupby(['fold','model','k']):
    rows.append(dict(fold=fold,model=model,k=k,up3=float((g.next_return>.03).mean()),down3=float((g.next_return<-.03).mean()),up5=float((g.next_return>.05).mean()),down5=float((g.next_return<-.05).mean()),up10=float((g.next_return>.10).mean()),down10=float((g.next_return<-.10).mean()),payout_ratio=float(g.next_return[g.next_return>0].mean()/-g.next_return[g.next_return<0].mean()) if (g.next_return>0).any() and (g.next_return<0).any() else float('nan')))
hit=pd.DataFrame(rows)
hit['net_tail_5']=hit.up5-hit.down5
hit['total_tail_5']=hit.up5+hit.down5
display(hit.loc[hit.model.isin(selected_names)].sort_values(['model','fold']).round(5))

## Experiments 74–78 · Quantile consistency and expected-shortfall proxies (diagnostic only)

In [ ]:
quantiles=['q05','q10','q25','q50','q75','q90','q95']
q=scored[quantiles].to_numpy(float)
violations=np.any(np.diff(q,axis=1)<0,axis=1)
print('Quantile crossing share:',float(violations.mean()))
# These are realized diagnostics, NOT ex-ante position filters.
for model in ['ridge_all','q90_minus_q10_risk','weighted_tail_utility']:
    g=trades.loc[(trades.model==model)&trades['k'].eq(5)]
    for fold,h in g.groupby('fold'):
        cutoff=h.next_return.quantile(.05)
        es=float(h.loc[h.next_return<=cutoff,'next_return'].mean())
        print(model,'fold',fold,'realized 5% expected-shortfall proxy:',round(es,5))

## Experiments 79–83 · Exploratory decision and reproducible exports

In [ ]:
decision=leader.loc[leader['k'].eq(5)].merge(paired_summary.rename(columns={'positive_folds':'positive_lift_folds'}),on='model',how='left',validate='one_to_one')
decision['exploratory_only']=(decision['positive_folds'].eq(3))&(decision.market_excess>0)&(decision.profit_factor>1)
display(decision.sort_values('mean_daily',ascending=False).head(25).round(5))
save_tables(OUT,metrics=metrics,daily=daily,trades=trades,events=ev,calibration=calibration,paired=paired,concentration=tails,tail_hits=hit,decision=decision,audit=results['audit'],timings=results['timings'],baseline_reconciliation=reconcile)
print('Saved to',OUT)
print('Interpretation: These are exploratory development-fold comparisons, not independently validated discoveries. No final holdout opened.')